<div style='background:#0f172a;padding:28px 32px;border-radius:12px;color:#e2e8f0;font-family:system-ui,Arial'>
<h1 style='margin:0;font-size:22px'>Registro — BraTS 2024 GLI · información mutua + rígido</h1>
<p style='color:#94a3b8;margin:8px 0 0'>Lee las salidas del EDA (<code>parametros_registro.json</code>, <code>casos_demostrativos.csv</code>) y aplica el motor del Taller 4 (Mattes MI + Euler3D, multi-resolución).<br>
Como el EDA concluyó que las modalidades ya vienen coregistradas, el registro es <b>demostrativo</b>: perturbación rígida conocida → recuperación (TRE).<br>
Salidas en <code>final-project/registro/</code>.</p></div>

In [ ]:
# === Setup (SimpleITK + nibabel) ===
import importlib, subprocess, sys
for pkg, mod in [('SimpleITK','SimpleITK'), ('nibabel','nibabel')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install',pkg,'--quiet'])
import os, glob, gc, json, shutil, zipfile, time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
print('SimpleITK', sitk.Version.VersionString())

In [ ]:
# === Fuente de datos (Drive) — robusta ===
try:
    from google.colab import drive
    drive.mount('/content/drive'); EN_COLAB = True
except Exception:
    EN_COLAB = False
if EN_COLAB:
    try: os.listdir('/content/drive/MyDrive')
    except Exception:
        from google.colab import drive; drive.mount('/content/drive', force_remount=True)

# >>> AJUSTA SI HACE FALTA <<<
DRIVE_DIR = '/content/drive/MyDrive/BRATS-2024'

def _es_entrenamiento(z):
    n = os.path.basename(z).lower(); return ('training' in n) and ('validation' not in n)
def _encontrar_zip(d):
    cands = sorted(glob.glob(os.path.join(d, '*.zip')))
    if not cands and os.path.isdir('/content/drive/MyDrive'):
        cands = sorted(glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True))
    ent = [z for z in cands if _es_entrenamiento(z)]
    if ent:
        principal = [z for z in ent if 'additional' not in os.path.basename(z).lower()]
        return (principal or ent)[0]
    return cands[0] if cands else None
TRAINING_ZIP = _encontrar_zip(DRIVE_DIR)
print('ZIP:', TRAINING_ZIP)
if TRAINING_ZIP is None:
    raise FileNotFoundError('No se encontró el ZIP de entrenamiento en Drive. Revisa DRIVE_DIR/montaje.')

BASE_PROY = os.path.join(DRIVE_DIR, 'final-project') if os.path.isdir(DRIVE_DIR) else 'final-project'
EDA_DIR = os.path.join(BASE_PROY, 'eda')
REG_DIR = os.path.join(BASE_PROY, 'registro')
FIG_DIR = os.path.join(REG_DIR, 'figuras'); VOL_DIR = os.path.join(REG_DIR, 'volumenes'); TFM_DIR = os.path.join(REG_DIR, 'transformaciones')
for d in [REG_DIR, FIG_DIR, VOL_DIR, TFM_DIR]: os.makedirs(d, exist_ok=True)
TMP_DIR = '/content/_reg_tmp'; os.makedirs(TMP_DIR, exist_ok=True)

def _buscar_eda(nombre):
    p = os.path.join(EDA_DIR, nombre)
    if os.path.exists(p): return p
    h = glob.glob(os.path.join(DRIVE_DIR, '**', nombre), recursive=True)
    return h[0] if h else None
print('Entradas EDA:', EDA_DIR, '| Salidas registro:', REG_DIR)

In [ ]:
# === Entradas del EDA: parámetros y casos ===
PARAMS_JSON = _buscar_eda('parametros_registro.json')
CASOS_CSV   = _buscar_eda('casos_demostrativos.csv')
PARAMS = {}
if PARAMS_JSON and os.path.exists(PARAMS_JSON):
    with open(PARAMS_JSON) as f: PARAMS = json.load(f)

MODALIDAD = PARAMS.get('modalidad_fija_recomendada', 'T1n').lower()
if MODALIDAD not in ['t1n','t1c','t2w','t2f']: MODALIDAD = 't1c'
NECESITA_INTRA = bool(PARAMS.get('registro_intra_sujeto_necesario', False))

if CASOS_CSV and os.path.exists(CASOS_CSV):
    REG_CASES_ALL = pd.read_csv(CASOS_CSV)['case_id'].astype(str).tolist()
else:
    REG_CASES_ALL = PARAMS.get('casos_demostrativos', [])

print('Modalidad fija:', MODALIDAD.upper())
print('¿Coregistro intra-sujeto necesario?:', NECESITA_INTRA)
print('Modo:', 'coregistro real moving->fixed' if NECESITA_INTRA else 'demostrativo (perturbación -> recuperación)')
print('Casos demostrativos disponibles:', REG_CASES_ALL)

In [ ]:
# === Configuración del registro (del Taller 4, optimizada para velocidad) ===
MAX_CASOS_REG = 3                 # nº de casos a registrar (cada uno ~1-3 min)
REG_CASES = REG_CASES_ALL[:MAX_CASOS_REG]

# Motor: Mattes MI + rígido Euler3D, multi-resolución
NBINS_MI = 50
MUESTREO = 0.1                    # fracción de voxeles muestreados por iteración
ITERS    = 150
SHRINK   = [4, 2]                 # hasta 2 mm (~10x más rápido, misma calidad que [4,2,1])
SIGMAS   = [2, 1]

# Perturbación rígida CONOCIDA (ground truth para el modo demostrativo)
ROT_DEG  = (6.0, 4.0, -5.0)       # grados (x,y,z)
TRA_MM   = (5.0, -4.0, 3.0)       # mm (x,y,z)
print('Casos a registrar:', REG_CASES)

In [ ]:
# === Utilidades de registro ===
def cargar_sitk_caso(case_id, mod):
    """Extrae <case_id>-<mod>.nii.gz del ZIP a TMP y lo lee como imagen sitk float32."""
    suf = f'{case_id}-{mod}.nii.gz'
    with zipfile.ZipFile(TRAINING_ZIP) as zf:
        miembro = next((n for n in zf.namelist() if n.endswith(suf)), None)
        if miembro is None: return None
        dest = os.path.join(TMP_DIR, os.path.basename(miembro))
        if not os.path.exists(dest):
            with zf.open(miembro) as s, open(dest, 'wb') as d: shutil.copyfileobj(s, d)
    return sitk.ReadImage(dest, sitk.sitkFloat32)

def perturbar(fixed):
    """Aplica una transformación rígida conocida y devuelve (transform, volumen_desalineado)."""
    centro = fixed.TransformContinuousIndexToPhysicalPoint([(s-1)/2 for s in fixed.GetSize()])
    p = sitk.Euler3DTransform(); p.SetCenter(centro)
    p.SetRotation(*[np.deg2rad(d) for d in ROT_DEG]); p.SetTranslation(TRA_MM)
    moving = sitk.Resample(fixed, fixed, p, sitk.sitkLinear, 0.0)
    return p, moving

def registrar(fixed, moving):
    """Registro rígido por información mutua (Mattes). Devuelve transform, curva, métrica, iteraciones."""
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=NBINS_MI)
    R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(MUESTREO, seed=42)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetOptimizerAsGradientDescentLineSearch(learningRate=1.0, numberOfIterations=ITERS,
            convergenceMinimumValue=1e-7, convergenceWindowSize=20)
    R.SetOptimizerScalesFromPhysicalShift()
    ini = sitk.CenteredTransformInitializer(fixed, moving, sitk.Euler3DTransform(),
            sitk.CenteredTransformInitializerFilter.GEOMETRY)
    R.SetInitialTransform(ini, inPlace=False)
    R.SetShrinkFactorsPerLevel(SHRINK); R.SetSmoothingSigmasPerLevel(SIGMAS)
    R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    curva = []
    R.AddCommand(sitk.sitkIterationEvent, lambda: curva.append(R.GetMetricValue()))
    final = R.Execute(fixed, moving)
    return final, curva, R.GetMetricValue(), R.GetOptimizerIteration()

def dice_cerebro(a, b):
    A = sitk.GetArrayFromImage(a) > 0; B = sitk.GetArrayFromImage(b) > 0
    s = A.sum() + B.sum(); return float(2*(A & B).sum()/s) if s else 0.0
def mse(a, b):
    A = sitk.GetArrayFromImage(a); B = sitk.GetArrayFromImage(b); return float(np.mean((A-B)**2))
def tre_mm(fixed, perturb, final):
    """Residual al componer la perturbación con la transform recuperada (~0 si recupera bien)."""
    S = fixed.GetSize()
    idxs = [[S[0]*f0, S[1]*f1, S[2]*f2] for f0,f1,f2 in
            [(.25,.25,.25),(.75,.25,.5),(.25,.75,.5),(.5,.5,.75),(.75,.75,.25),(.5,.25,.25)]]
    pts = [fixed.TransformContinuousIndexToPhysicalPoint([float(x) for x in idx]) for idx in idxs]
    return float(np.mean([np.linalg.norm(
        np.array(final.TransformPoint(perturb.TransformPoint(p))) - np.array(p)) for p in pts]))
def _axial_medio(img):
    a = sitk.GetArrayFromImage(img); return a[a.shape[0]//2]
print('Utilidades listas.')

---
## Registro de los casos demostrativos
Por cada caso: se carga la modalidad fija, se perturba con la transformación rígida conocida, se registra para recuperarla y se miden TRE, Dice de cerebro y MSE (antes/después). Se guardan volúmenes, transformaciones y figuras.

In [ ]:
# === Ejecutar registro ===
filas, curvas = [], {}
for cid in REG_CASES:
    fixed = cargar_sitk_caso(cid, MODALIDAD)
    if fixed is None:
        print('  sin volumen para', cid); continue
    if NECESITA_INTRA:
        moving_mod = next((m for m in ['t1n','t1c','t2w','t2f'] if m != MODALIDAD), MODALIDAD)
        moving = cargar_sitk_caso(cid, moving_mod); perturb = None
    else:
        perturb, moving = perturbar(fixed)

    dice_antes, mse_antes = dice_cerebro(fixed, moving), mse(fixed, moving)
    t0 = time.time()
    final, curva, metrica, n_it = registrar(fixed, moving)
    recuperado = sitk.Resample(moving, fixed, final, sitk.sitkLinear, 0.0)
    curvas[cid] = curva

    fila = {'case_id': cid,
            'dice_cerebro_antes': round(dice_antes, 4),
            'dice_cerebro_despues': round(dice_cerebro(fixed, recuperado), 4),
            'mse_antes': round(mse_antes, 5), 'mse_despues': round(mse(fixed, recuperado), 5),
            'iteraciones': int(n_it), 'metrica_MI_final': round(float(metrica), 5),
            'segundos': round(time.time()-t0, 1)}
    if perturb is not None:
        fila['TRE_mm'] = round(tre_mm(fixed, perturb, final), 4)
    filas.append(fila)

    sitk.WriteImage(moving,     os.path.join(VOL_DIR, f'{cid}-desalineado.nii.gz'))
    sitk.WriteImage(recuperado, os.path.join(VOL_DIR, f'{cid}-recuperado.nii.gz'))
    sitk.WriteTransform(final,  os.path.join(TFM_DIR, f'{cid}-transform.tfm'))

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    for ax, (t, im) in zip(axes, [('Fija', fixed), ('Desalineada', moving), ('Recuperada', recuperado)]):
        ax.imshow(_axial_medio(im), cmap='gray', origin='lower', aspect='auto')
        ax.set_title(t); ax.axis('off')
    fig.suptitle(f'Registro — {cid} · {MODALIDAD.upper()}')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, f'registro_{cid}.png'), dpi=130, bbox_inches='tight'); plt.show()
    print('  registrado:', cid, fila)

df_reg = pd.DataFrame(filas)
df_reg.to_csv(os.path.join(REG_DIR, 'metricas_registro.csv'), index=False)
with open(os.path.join(REG_DIR, 'parametros_usados.json'), 'w', encoding='utf-8') as f:
    json.dump({'modalidad': MODALIDAD, 'necesita_intra': NECESITA_INTRA, 'NBINS_MI': NBINS_MI,
               'MUESTREO': MUESTREO, 'ITERS': ITERS, 'SHRINK': SHRINK, 'SIGMAS': SIGMAS,
               'ROT_DEG': ROT_DEG, 'TRA_MM': TRA_MM}, f, ensure_ascii=False, indent=2)
print('metricas_registro.csv y parametros_usados.json guardados.')
display(df_reg)

In [ ]:
# === Curva de convergencia (métrica MI por iteración) ===
if curvas:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for cid, c in curvas.items(): ax.plot(range(len(c)), c, label=cid, linewidth=1.6)
    ax.set_xlabel('Iteración'); ax.set_ylabel('Mattes MI (menor = mejor)')
    ax.set_title('Convergencia del registro'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'convergencia_registro.png'), dpi=130, bbox_inches='tight'); plt.show()
    print('Curva de convergencia guardada.')

---
## Reporte HTML
Genera un `report.html` autocontenido (figuras embebidas en base64) con el mismo estilo que el reporte de segmentación: KPIs, método, configuración, métricas por caso, resultados visuales, convergencia y discusión. Se guarda en `final-project/registro/` y queda incluido en el ZIP de salidas.

In [ ]:
# === Generar report.html (autocontenido, mismo estilo que el reporte de segmentación) ===
import base64, html as _html

def _img_b64(path):
    """Devuelve un data-URI base64 del PNG; cadena vacía si no existe."""
    if path and os.path.exists(path):
        with open(path, 'rb') as f:
            return 'data:image/png;base64,' + base64.b64encode(f.read()).decode('ascii')
    return ''

def _fig(path, alt=''):
    b = _img_b64(path)
    if not b:
        return f'<div class="fig note">[figura no encontrada: {_html.escape(os.path.basename(path))}]</div>'
    return f'<div class="fig"><img src="{b}" alt="{_html.escape(alt)}" style="width:100%"></div>'

def _tabla(df, columnas=None, etiquetas=None):
    """DataFrame -> tabla HTML con la clase 'dataframe' (estilo pandas)."""
    if df is None or len(df) == 0:
        return '<p class="note">Sin datos.</p>'
    d = df.copy()
    if columnas:
        columnas = [c for c in columnas if c in d.columns]
        d = d[columnas]
    if etiquetas:
        d = d.rename(columns=etiquetas)
    return d.to_html(index=False, classes='dataframe', border=0, justify='right')

# --- Resultados: memoria si existe, si no desde disco ---
try:
    df_rep = df_reg.copy()
except NameError:
    _csv = os.path.join(REG_DIR, 'metricas_registro.csv')
    df_rep = pd.read_csv(_csv) if os.path.exists(_csv) else pd.DataFrame()

_par_path = os.path.join(REG_DIR, 'parametros_usados.json')
PAR = {}
if os.path.exists(_par_path):
    with open(_par_path) as f: PAR = json.load(f)

modalidad = str(PAR.get('modalidad', MODALIDAD if 'MODALIDAD' in dir() else 't1c')).upper()
necesita_intra = bool(PAR.get('necesita_intra', NECESITA_INTRA if 'NECESITA_INTRA' in dir() else False))
modo_txt = 'coregistro real moving->fixed' if necesita_intra else 'demostrativo (perturbación rígida conocida -> recuperación)'

# --- KPIs ---
n_casos = len(df_rep)
hay_tre  = ('TRE_mm' in df_rep.columns) and n_casos
tre_min  = float(df_rep['TRE_mm'].min())  if hay_tre else None
tre_med  = float(df_rep['TRE_mm'].mean()) if hay_tre else None
dice_med = float(df_rep['dice_cerebro_despues'].mean()) if ('dice_cerebro_despues' in df_rep.columns and n_casos) else None
seg_med  = float(df_rep['segundos'].mean()) if ('segundos' in df_rep.columns and n_casos) else None

kpi1 = (f'{tre_min:.3f} mm', 'Mejor TRE — residual') if tre_min is not None else ('—', 'TRE no aplicable')
kpi2 = (str(n_casos), 'Casos registrados')
kpi3 = (f'{dice_med:.3f}', 'Dice cerebro medio — después') if dice_med is not None else ('—', 'Dice cerebro')
kpi4 = (modalidad, 'Modalidad fija')

# --- Parámetros del motor (tabla) ---
_pares = [
    ('Métrica', 'Mattes Mutual Information'),
    ('Transformación', 'Euler3D (rígida: 3 rotaciones + 3 traslaciones)'),
    ('Bins del histograma (MI)', PAR.get('NBINS_MI', '—')),
    ('Muestreo aleatorio', f"{PAR.get('MUESTREO', '—')} (fracción de vóxeles/iteración, seed=42)"),
    ('Iteraciones máx.', PAR.get('ITERS', '—')),
    ('Multi-resolución — shrink', PAR.get('SHRINK', '—')),
    ('Multi-resolución — sigmas', PAR.get('SIGMAS', '—')),
    ('Optimizador', 'Gradient Descent Line Search, escalas por physical shift'),
    ('Inicialización', 'CenteredTransformInitializer (GEOMETRY)'),
]
if not necesita_intra:
    _pares += [
        ('Rotación perturbada (x,y,z)°', PAR.get('ROT_DEG', '—')),
        ('Traslación perturbada (x,y,z) mm', PAR.get('TRA_MM', '—')),
    ]
_filas_par = ''.join(
    f'<tr><td>{_html.escape(str(k))}</td><td>{_html.escape(str(v))}</td></tr>' for k, v in _pares)
tabla_par = (f'<table class="dataframe"><thead><tr><th>Parámetro</th>'
             f'<th>Valor</th></tr></thead><tbody>{_filas_par}</tbody></table>')

# --- Tabla de métricas (formateada) ---
_cols = ['case_id', 'TRE_mm', 'dice_cerebro_antes', 'dice_cerebro_despues',
         'mse_antes', 'mse_despues', 'metrica_MI_final', 'iteraciones', 'segundos']
_lab = {'case_id':'Caso', 'TRE_mm':'TRE (mm)',
        'dice_cerebro_antes':'Dice antes', 'dice_cerebro_despues':'Dice después',
        'mse_antes':'MSE antes', 'mse_despues':'MSE después',
        'metrica_MI_final':'MI final', 'iteraciones':'Iter.', 'segundos':'Tiempo (s)'}
tabla_met = _tabla(df_rep, _cols, _lab)

# --- Figuras por caso + convergencia ---
casos_ids = df_rep['case_id'].astype(str).tolist() if 'case_id' in df_rep.columns else []
secciones_figs = ''
for cid in casos_ids:
    fp = os.path.join(FIG_DIR, f'registro_{cid}.png')
    secciones_figs += f'<h3>{_html.escape(cid)}</h3>' + _fig(fp, f'Registro {cid}: fija / desalineada / recuperada')
fig_conv = _fig(os.path.join(FIG_DIR, 'convergencia_registro.png'), 'Convergencia de la métrica MI')

# --- Resumen ejecutivo dinámico ---
if hay_tre:
    resumen = (f'Se realizó <b>registro rígido por información mutua</b> (Mattes MI + Euler3D, '
               f'multi-resolución) sobre <b>{n_casos}</b> caso(s) de BraTS 2024 GLI en la modalidad '
               f'<b>{modalidad}</b>. Como el EDA concluyó que las modalidades ya vienen coregistradas, '
               f'el ejercicio es <b>demostrativo</b>: se aplica una perturbación rígida conocida y se '
               f'mide la capacidad del motor para recuperarla. El <b>TRE</b> (error residual sobre puntos '
               f'de control) descendió a un mínimo de <b>{tre_min:.3f} mm</b> '
               f'(media {tre_med:.3f} mm), confirmando que el pipeline recupera con precisión '
               f'sub-vóxel la transformación impuesta.')
else:
    resumen = (f'Se realizó <b>coregistro intra-sujeto</b> (Mattes MI + Euler3D, multi-resolución) sobre '
               f'<b>{n_casos}</b> caso(s) de BraTS 2024 GLI, tomando <b>{modalidad}</b> como modalidad fija. '
               f'Se reportan Dice de cerebro y MSE antes/después del registro.')

dice_txt = f'{dice_med:.3f}' if dice_med is not None else 'n/d'
seg_txt  = f'{seg_med:.1f}'  if seg_med  is not None else 'n/d'

# --- HTML ---
HTML = f"""<!doctype html><html lang="es"><head><meta charset="utf-8"><title>Registro BraTS 2024 GLI — reporte</title><style>
:root{{--azul:#1f77b4;--naranja:#ff7f0e;--morado:#9467bd;--verde:#2ca02c;--rojo:#d62728;
 --tinta:#1f2a37;--suave:#5b6b7c;--linea:#e3e8ef;--fondo:#f6f8fb}}
*{{box-sizing:border-box}}
body{{font-family:'Segoe UI',system-ui,Arial,sans-serif;margin:0;color:var(--tinta);background:var(--fondo);line-height:1.55}}
.wrap{{max-width:1180px;margin:0 auto;padding:0 22px 60px}}
header.hero{{background:linear-gradient(120deg,#1e3c72,#2a5298 60%,#3a7bd5);color:#fff;padding:38px 22px 30px}}
header.hero h1{{margin:0 0 6px;font-size:26px}} header.hero p{{margin:0;color:#dce6f7;max-width:860px}}
.kpis{{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:20px 0}}
.kpi{{background:#fff;border-radius:12px;padding:14px 16px;box-shadow:0 1px 3px rgba(20,40,80,.08);border-top:4px solid var(--azul)}}
.kpi .v{{font-size:23px;font-weight:700}} .kpi .l{{color:var(--suave);font-size:13px}}
.kpi.l2{{border-color:var(--morado)}} .kpi.l3{{border-color:var(--verde)}} .kpi.l4{{border-color:var(--naranja)}}
h2{{margin:36px 0 6px;font-size:20px;border-left:5px solid var(--azul);padding-left:12px}}
h3{{margin:16px 0 6px;font-size:15px}} p{{max-width:980px}}
.grid2{{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:14px 0}}
.card{{background:#fff;border-radius:12px;padding:14px 16px;box-shadow:0 1px 3px rgba(20,40,80,.07);border:1px solid var(--linea)}}
.card.m{{border-top:5px solid var(--azul)}} .card.pm{{border-left:4px solid var(--naranja);background:#fff8f1}}
.fig{{background:#fff;border:1px solid var(--linea);border-radius:12px;padding:8px;margin:12px 0}}
table{{border-collapse:collapse;margin:8px 0;width:100%;background:#fff;border-radius:10px;overflow:hidden;box-shadow:0 1px 3px rgba(20,40,80,.06)}}
th,td{{border-bottom:1px solid var(--linea);padding:7px 12px;text-align:right;font-size:13px}}
th{{background:#eef2f8;color:#2a3f5f}} td:first-child,th:first-child{{text-align:left}}
tbody tr:nth-child(even){{background:#fafbfd}}
.note{{color:var(--suave);font-size:13px}}
</style></head><body>
<header class="hero"><div class="wrap">
<h1>Registro rígido por información mutua — BraTS 2024 GLI</h1>
<p>Motor del Taller 4 (Mattes MI + Euler3D, multi-resolución) · modo {modo_txt} · métricas TRE, Dice de cerebro y MSE, curva de convergencia.</p>
</div></header>
<div class="wrap">

<div class="kpis">
<div class="kpi"><div class="v">{kpi1[0]}</div><div class="l">{kpi1[1]}</div></div>
<div class="kpi l2"><div class="v">{kpi2[0]}</div><div class="l">{kpi2[1]}</div></div>
<div class="kpi l3"><div class="v">{kpi3[0]}</div><div class="l">{kpi3[1]}</div></div>
<div class="kpi l4"><div class="v">{kpi4[0]}</div><div class="l">{kpi4[1]}</div></div>
</div>

<h2>1. Resumen ejecutivo</h2>
<p>{resumen}</p>

<h2>2. Método</h2>
<div class="grid2">
<div class="card m"><h4>Mattes Mutual Information</h4><p class="note">Métrica robusta para alineación multimodal; no asume relación lineal de intensidades. Se estima sobre un muestreo aleatorio de vóxeles por iteración para acelerar el cálculo.</p></div>
<div class="card m"><h4>Transformación rígida (Euler3D)</h4><p class="note">Seis grados de libertad (3 rotaciones + 3 traslaciones). Inicializada por centro geométrico con CenteredTransformInitializer.</p></div>
<div class="card m"><h4>Multi-resolución</h4><p class="note">Pirámide con factores de reducción y suavizado gaussiano decrecientes; evita mínimos locales y reduce el tiempo de cómputo sin perder calidad.</p></div>
<div class="card m"><h4>Validación demostrativa (TRE)</h4><p class="note">Se aplica una transformación rígida conocida (ground truth) y se mide el error residual al recuperarla sobre puntos de control distribuidos en el volumen.</p></div>
</div>

<h2>3. Configuración del registro</h2>
{tabla_par}

<h2>4. Métricas — por caso</h2>
{tabla_met}
<p class="note">TRE: error residual medio (mm) tras componer perturbación y transformación recuperada (≈0 = recuperación correcta). Dice de cerebro y MSE comparan el volumen fijo con el desalineado (antes) y con el recuperado (después). MI final: valor de la métrica de Mattes al converger (menor = mejor).</p>

<h2>5. Resultados visuales por caso</h2>
<p class="note">Cada panel muestra el corte axial medio: <b>fija</b> · <b>desalineada</b> (tras la perturbación) · <b>recuperada</b> (tras el registro).</p>
{secciones_figs}

<h2>6. Convergencia</h2>
<p class="note">Evolución de la métrica de Mattes MI por iteración (menor = mejor alineación).</p>
{fig_conv}

<h2>7. Discusión y limitaciones</h2>
<div class="card pm"><p>Al venir las modalidades ya coregistradas en BraTS, el registro real moving→fixed sería casi la identidad y poco informativo. Por eso se adopta un esquema <b>demostrativo</b>: una perturbación rígida conocida permite medir objetivamente la precisión del motor (TRE). El método es <b>rígido</b>, por lo que solo corrige rotaciones y traslaciones globales; deformaciones locales o cambios de escala requerirían etapas afín/BSpline. El Dice de cerebro es una verificación de solapamiento global y no refleja precisión a nivel de estructuras finas.</p></div>

<h2>8. Reproducibilidad</h2>
<p class="note">Modalidad fija: {modalidad} · modo: {modo_txt} · motor: Mattes MI + Euler3D multi-resolución (shrink {PAR.get('SHRINK','—')}, sigmas {PAR.get('SIGMAS','—')}, {PAR.get('NBINS_MI','—')} bins, muestreo {PAR.get('MUESTREO','—')} seed=42, {PAR.get('ITERS','—')} iter máx.) · Dice medio después: {dice_txt} · tiempo medio/caso: {seg_txt} s · salidas en final-project/registro/.</p>

</div></body></html>"""

_report_path = os.path.join(REG_DIR, 'report.html')
with open(_report_path, 'w', encoding='utf-8') as f:
    f.write(HTML)
print('report.html generado en:', _report_path, '·', round(os.path.getsize(_report_path)/1024, 1), 'KB')

# Mostrarlo en línea si estamos en notebook
try:
    from IPython.display import IFrame, display, HTML as _IPHTML
    display(_IPHTML(f'<b>report.html</b> guardado en <code>{_report_path}</code>'))
except Exception:
    pass


In [ ]:
# === Empaquetar y descargar salidas del registro ===
try:
    from google.colab import files; _COLAB = True
except Exception:
    _COLAB = False
zip_out = '/content/registro_salidas.zip' if EN_COLAB else 'registro_salidas.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(REG_DIR):
        for fn in fnames:
            fp = os.path.join(root, fn); zf.write(fp, os.path.relpath(fp, BASE_PROY))
print('ZIP:', zip_out, round(os.path.getsize(zip_out)/1e6, 2), 'MB')
if _COLAB: files.download(zip_out)